# Part B: Practical Work with GUDHI

Authors: Naama Skital and Adir Ofir


In [ ]:
%pip install -q gudhi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gudhi

In [ ]:
def compute_rips(points, max_edge_length):
    rips = gudhi.RipsComplex(
        points=points,
        max_edge_length=max_edge_length
    )

    simplex_tree = rips.create_simplex_tree(
        max_dimension=2
    )

    persistence = simplex_tree.persistence(
        homology_coeff_field=2
    )

    h0 = simplex_tree.persistence_intervals_in_dimension(0)
    h1 = simplex_tree.persistence_intervals_in_dimension(1)

    return simplex_tree, persistence, h0, h1


def count_simplices(points, max_edge_length):
    rips = gudhi.RipsComplex(
        points=points,
        max_edge_length=max_edge_length
    )

    simplex_tree = rips.create_simplex_tree(
        max_dimension=2
    )

    return simplex_tree.num_simplices()

# Task 1: Reproducing the Hand Calculation

We construct the filtered complex from Questions 6 and 7 and compute its persistent homology.

In [ ]:
simplex_tree = gudhi.SimplexTree()

for vertex in [0, 1, 2, 3]:
    simplex_tree.insert(
        [vertex],
        filtration=0
    )

edges = [
    ([0, 1], 1),
    ([1, 2], 2),
    ([2, 3], 3),
    ([3, 0], 4),
    ([0, 2], 5)
]

for edge, time in edges:
    simplex_tree.insert(
        edge,
        filtration=time
    )

triangles = [
    ([0, 1, 2], 6),
    ([0, 2, 3], 7)
]

for triangle, time in triangles:
    simplex_tree.insert(
        triangle,
        filtration=time
    )

simplex_tree.make_filtration_non_decreasing()

for simplex, time in simplex_tree.get_filtration():
    print(simplex, time)

In [ ]:
task1_persistence = simplex_tree.persistence(
    homology_coeff_field=2
)

task1_h0 = simplex_tree.persistence_intervals_in_dimension(0)
task1_h1 = simplex_tree.persistence_intervals_in_dimension(1)

print("H0 intervals:")
print(task1_h0)

print("\nH1 intervals:")
print(task1_h1)

## Persistence Diagram

In [ ]:
gudhi.plot_persistence_diagram(
    task1_persistence
)

plt.title("Task 1: Persistence Diagram")
plt.show()

## Persistence Barcode

In [ ]:
gudhi.plot_persistence_barcode(
    task1_persistence
)

plt.title("Task 1: Persistence Barcode")
plt.show()

## Comparison with the Hand Calculation

The GUDHI result agrees with the hand calculation.

In H0, three components die at times 1, 2 and 3, and one component remains.

In H1, one cycle exists from time 4 to time 7, and another cycle exists from time 5 to time 6.

GUDHI uses `inf` for a class that does not die. The intervals may appear in a different order.

# Task 2: Synthetic Point Clouds

We generate four point clouds with fixed random seeds and compute persistent homology in dimensions 0 and 1.

In [ ]:
def analyze_points(points, title, max_edge_length):
    plt.figure(figsize=(5, 5))
    plt.scatter(points[:, 0], points[:, 1], s=15)
    plt.title(title)
    plt.axis("equal")
    plt.show()

    simplex_tree, persistence, h0, h1 = compute_rips(
        points,
        max_edge_length
    )

    print("H0 intervals:")
    print(h0)

    print("\nH1 intervals:")
    print(h1)

    gudhi.plot_persistence_diagram(persistence)
    plt.title(title + ": Persistence Diagram")
    plt.show()

    return simplex_tree, h0, h1

## Noisy Circle

We generate 80 points on a circle.

We add Gaussian noise with standard deviation 0.05 and use seed 101.

In [ ]:
rng = np.random.default_rng(101)

angles = rng.uniform(
    0,
    2 * np.pi,
    80
)

circle_points = np.column_stack(
    (
        np.cos(angles),
        np.sin(angles)
    )
)

circle_points += rng.normal(
    0,
    0.05,
    circle_points.shape
)

circle_tree, circle_h0, circle_h1 = analyze_points(
    circle_points,
    "Noisy Circle",
    2.0
)

### Interpretation

There is one long H1 interval. It represents the hole inside the circle.

The short intervals are caused by noise and by the finite number of points.

## Noisy Figure-Eight

We generate 100 points in a figure-eight shape.

We add Gaussian noise with standard deviation 0.04 and use seed 202.

In [ ]:
rng = np.random.default_rng(202)

t = rng.uniform(
    0,
    2 * np.pi,
    100
)

figure_eight_points = np.column_stack(
    (
        np.sin(t),
        np.sin(t) * np.cos(t)
    )
)

figure_eight_points += rng.normal(
    0,
    0.04,
    figure_eight_points.shape
)

figure_eight_tree, figure_eight_h0, figure_eight_h1 = (
    analyze_points(
        figure_eight_points,
        "Noisy Figure-Eight",
        2.0
    )
)

### Interpretation

There are two long H1 intervals. They represent the two holes of the figure-eight.

The short intervals are caused by noise and uneven distances between the points.

## Two Separated Noisy Circles

We generate two circles with 50 points on each circle.

Each circle has radius 0.75. The distance between the centers is 3.0.

We add Gaussian noise with standard deviation 0.04 and use seed 303.

In [ ]:
rng = np.random.default_rng(303)

angles_1 = rng.uniform(
    0,
    2 * np.pi,
    50
)

angles_2 = rng.uniform(
    0,
    2 * np.pi,
    50
)

circle_1 = np.column_stack(
    (
        -1.5 + 0.75 * np.cos(angles_1),
        0.75 * np.sin(angles_1)
    )
)

circle_2 = np.column_stack(
    (
        1.5 + 0.75 * np.cos(angles_2),
        0.75 * np.sin(angles_2)
    )
)

separated_points = np.vstack(
    (
        circle_1,
        circle_2
    )
)

separated_points += rng.normal(
    0,
    0.04,
    separated_points.shape
)

separated_tree, separated_h0, separated_h1 = (
    analyze_points(
        separated_points,
        "Two Separated Noisy Circles",
        2.0
    )
)

### Interpretation

At first, the two circles form two connected components.

There are two long H1 intervals. They represent the two holes inside the circles.

The short intervals are caused by noise and by the finite number of points.

## Two Nested Noisy Circles

We generate two circles with the same center.

The inner radius is 0.55 and the outer radius is 1.25. Each circle contains 60 points.

We add Gaussian noise with standard deviation 0.035 and use seed 404.

In [ ]:
rng = np.random.default_rng(404)

inner_angles = rng.uniform(
    0,
    2 * np.pi,
    60
)

outer_angles = rng.uniform(
    0,
    2 * np.pi,
    60
)

inner_circle = np.column_stack(
    (
        0.55 * np.cos(inner_angles),
        0.55 * np.sin(inner_angles)
    )
)

outer_circle = np.column_stack(
    (
        1.25 * np.cos(outer_angles),
        1.25 * np.sin(outer_angles)
    )
)

nested_points = np.vstack(
    (
        inner_circle,
        outer_circle
    )
)

nested_points += rng.normal(
    0,
    0.035,
    nested_points.shape
)

nested_tree, nested_h0, nested_h1 = analyze_points(
    nested_points,
    "Two Nested Noisy Circles",
    2.3
)

### Interpretation

There is one clearly long H1 interval. The other intervals are shorter.

Although the data contains two circles, edges between the inner and outer circles appear early and affect the detected holes.

The short intervals are caused by noise and temporary cycles between the circles.

## Effect of the Maximum Edge Length

We compare the number of simplices for two maximum edge lengths.

In [ ]:
simplex_counts = pd.DataFrame(
    {
        "Dataset": [
            "Noisy circle",
            "Figure-eight",
            "Separated circles",
            "Nested circles"
        ],
        "Small edge length": [
            0.35,
            0.30,
            0.30,
            0.30
        ],
        "Small complex": [
            count_simplices(circle_points, 0.35),
            count_simplices(figure_eight_points, 0.30),
            count_simplices(separated_points, 0.30),
            count_simplices(nested_points, 0.30)
        ],
        "Large edge length": [
            2.0,
            2.0,
            2.0,
            2.3
        ],
        "Large complex": [
            count_simplices(circle_points, 2.0),
            count_simplices(figure_eight_points, 2.0),
            count_simplices(separated_points, 2.0),
            count_simplices(nested_points, 2.3)
        ]
    }
)

simplex_counts

A larger maximum edge length creates many more simplices.

A small value is faster, but it may stop before an important interval ends.

# Task 3: Stability Experiment

We use a clean circle with 80 points.

We add Gaussian noise with levels 0, 0.02, 0.05, 0.10, 0.15 and 0.20.

Each noise level is tested with five random seeds.

In [ ]:
noise_levels = [
    0,
    0.02,
    0.05,
    0.10,
    0.15,
    0.20
]

stability_seeds = [
    501,
    502,
    503,
    504,
    505
]

angles = np.linspace(
    0,
    2 * np.pi,
    80,
    endpoint=False
)

clean_points = np.column_stack(
    (
        np.cos(angles),
        np.sin(angles)
    )
)

_, _, _, clean_h1 = compute_rips(
    clean_points,
    2.0
)

plt.figure(figsize=(5, 5))
plt.scatter(
    clean_points[:, 0],
    clean_points[:, 1],
    s=15
)
plt.title("Clean Circle")
plt.axis("equal")
plt.show()

In [ ]:
stability_records = []

for sigma in noise_levels:
    for seed in stability_seeds:
        rng = np.random.default_rng(seed)

        noise = rng.normal(
            0,
            sigma,
            clean_points.shape
        )

        noisy_points = clean_points + noise

        _, _, _, noisy_h1 = compute_rips(
            noisy_points,
            2.0
        )

        distance = gudhi.bottleneck_distance(
            clean_h1,
            noisy_h1
        )

        if distance < 1e-12:
            distance = 0.0

        displacements = np.linalg.norm(
            noisy_points - clean_points,
            axis=1
        )

        max_displacement = displacements.max()

        stability_records.append(
            {
                "sigma": sigma,
                "seed": seed,
                "bottleneck distance": distance,
                "maximum displacement": max_displacement,
                "stability bound": 2 * max_displacement,
                "bound satisfied": (
                    distance
                    <= 2 * max_displacement + 1e-10
                )
            }
        )

stability_results = pd.DataFrame(
    stability_records
)

stability_results

In [ ]:
stability_summary = (
    stability_results
    .groupby("sigma")["bottleneck distance"]
    .agg(["mean", "std"])
    .reset_index()
)

stability_summary

## Mean Bottleneck Distance

In [ ]:
plt.figure(figsize=(7, 4))

plt.errorbar(
    stability_summary["sigma"],
    stability_summary["mean"],
    yerr=stability_summary["std"],
    marker="o",
    capsize=4
)

plt.xlabel("Noise level")
plt.ylabel("Mean bottleneck distance")
plt.title("Bottleneck Distance by Noise Level")
plt.grid(True)
plt.show()

In [ ]:
print(
    "All bounds satisfied:",
    stability_results["bound satisfied"].all()
)

## Interpretation

The mean bottleneck distance increased when the noise level increased.

This means that stronger noise caused larger changes in the H1 persistence diagram.

The error bars show the standard deviation over the five random seeds.

In all experiments, the bottleneck distance was smaller than twice the maximum point displacement. This agrees with the stability result from the course.

The stability result gives an upper bound, so the bottleneck distance does not have to be equal to the bound.

## Limitations

The circle contains only 80 points, so the result depends on the finite sample.

We used only five random seeds for each noise level. More repetitions could give more reliable results.

# Sources

We used the official GUDHI, NumPy and Matplotlib documentation.
